# Capa Gold — Agregacion en Ventanas de 15 Minutos

**Pipeline Medallon:** Bronze (ingesta) → Silver (limpieza) → **Gold (agregacion)**  
**Proyecto:** Prediccion de Demanda de Transporte Publico — ODS 11  
**Dataset:** UrbanBus — Registros AFC (Automated Fare Collection)  

---

## Objetivo de esta capa

La capa Gold transforma los datos limpios (Silver) en el dataset final listo para
alimentar modelos de Machine Learning. Las operaciones principales son:

1. **Binning temporal** en ventanas de 15 minutos (`dt.truncate`).
2. **Agregacion** del conteo de pasajeros por estacion y ventana temporal.
3. **Feature engineering** basico: extraccion de hora, dia de la semana, fecha.
4. **Validacion** contra las metricas del pipeline original para garantizar reproducibilidad.
5. **Exportacion** a Parquet y CSV.

### Justificacion del binning en ventanas de 15 minutos

La literatura reciente en prediccion de demanda de transporte publico ha convergido
en el uso de ventanas temporales de 15 minutos como la granularidad optima:

1. **Inviabilidad computacional de la prediccion por segundo:** Con millones de transacciones
   distribuidas en miles de estaciones, predecir la demanda segundo a segundo genera una
   matriz espacio-temporal de dimensionalidad intratable (Zou et al., 2022).

2. **Optimalidad operativa:** Las ventanas de 15 minutos se alinean con los intervalos
   tipicos de despacho de buses, lo que permite predicciones con valor operativo directo
   (Hao et al., 2019 — MGC-RNN; DST-TransitNet, 2023).

3. **Balance precision-generalizacion:** Ventanas menores (1-5 min) producen ruido
   estocastico excesivo; ventanas mayores (30-60 min) suavizan picos relevantes.
   Los 15 minutos son el compromiso optimo validado en multiples estudios con datos AFC
   (Liu et al., 2020; Short-term OD prediction in urban rail, 2019).

In [ ]:
# =============================================================================
# Importacion de bibliotecas
# =============================================================================

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'font.family': 'sans-serif',
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_style('whitegrid')

print(f"Polars version: {pl.__version__}")

In [ ]:
# =============================================================================
# Configuracion de rutas
# =============================================================================

PROJECT_ROOT = Path('..').resolve()
DATA_SILVER  = PROJECT_ROOT / 'data' / 'silver'
DATA_GOLD    = PROJECT_ROOT / 'data' / 'gold'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'  # Compatibilidad con pipeline anterior

# Entrada: Parquet limpio de la capa Silver
SILVER_INPUT = DATA_SILVER / 'bus_data_oct2017_clean.parquet'

# Salidas de esta capa
GOLD_PARQUET = DATA_GOLD / 'dataset_clean_15min.parquet'
GOLD_CSV     = DATA_GOLD / 'dataset_clean_15min.csv'

DATA_GOLD.mkdir(parents=True, exist_ok=True)

print(f"Entrada Silver:  {SILVER_INPUT}")
print(f"Existe:          {SILVER_INPUT.exists()}")
print(f"Salida Gold:     {DATA_GOLD}")

In [ ]:
# =============================================================================
# Metricas esperadas del pipeline original (para validacion)
# =============================================================================

EXPECTED_ROWS    = 506_545
EXPECTED_COLS    = 8
EXPECTED_TOP_STN = 'EJ_2'
EXPECTED_COLUMNS = [
    'Boarding_stop_stn', 'time_bin_15min', 'passenger_count',
    'unique_services', 'dominant_card_type', 'hour', 'day_of_week', 'date'
]

print("Metricas de referencia (pipeline original):")
print(f"  Filas esperadas:    {EXPECTED_ROWS:,}")
print(f"  Columnas esperadas: {EXPECTED_COLS}")
print(f"  Top estacion:       {EXPECTED_TOP_STN}")

In [ ]:
# =============================================================================
# Lectura del Parquet Silver
# =============================================================================

df_silver = pl.read_parquet(str(SILVER_INPUT))

print(f"Registros Silver: {df_silver.shape[0]:,}")
print(f"Columnas:         {df_silver.shape[1]}")

---
## 1. Binning Temporal — Ventanas de 15 Minutos

Se trunca el campo `ride_start_datetime` al intervalo de 15 minutos mas cercano
hacia abajo (floor). Ejemplos:

| Timestamp original | Ventana 15 min |
|-------------------|----------------|
| 08:07:23 | 08:00:00 |
| 08:16:45 | 08:15:00 |
| 08:29:59 | 08:15:00 |
| 08:30:01 | 08:30:00 |

In [ ]:
# =============================================================================
# 1. Creacion de la ventana temporal de 15 minutos
# =============================================================================

df_silver = df_silver.with_columns(
    pl.col('ride_start_datetime')
      .dt.truncate('15m')
      .alias('time_bin_15min')
)

# Verificar la transformacion
print("Muestra de la columna time_bin_15min:")
df_silver.select('ride_start_datetime', 'time_bin_15min').head(10)

---
## 2. Agregacion por Estacion y Ventana Temporal

Se cuenta el numero de transacciones (proxy de pasajeros) para cada combinacion
(estacion, ventana_temporal). Este es el formato requerido para alimentar modelos
de series temporales espaciales y redes de grafos espacio-temporales.

In [ ]:
# =============================================================================
# 2. Agregacion de demanda por estacion y ventana de 15 min
# =============================================================================

df_gold = (
    df_silver
    .group_by(['Boarding_stop_stn', 'time_bin_15min'])
    .agg(
        pl.len().alias('passenger_count'),                                 # Conteo de pasajeros
        pl.col('Bus_Service_Number').n_unique().alias('unique_services'),   # Lineas unicas
        pl.col('Card_Type').mode().first().alias('dominant_card_type'),     # Tarjeta dominante
    )
    .sort(['Boarding_stop_stn', 'time_bin_15min'])
)

print(f"Filas tras agregacion:  {df_gold.shape[0]:,}")
print(f"Columnas:               {df_gold.shape[1]}")

In [ ]:
# =============================================================================
# 3. Feature engineering — variables temporales derivadas
# =============================================================================

df_gold = df_gold.with_columns([
    pl.col('time_bin_15min').dt.hour().alias('hour'),            # Hora del dia
    pl.col('time_bin_15min').dt.weekday().alias('day_of_week'),  # Dia de la semana
    pl.col('time_bin_15min').dt.date().alias('date'),            # Fecha
])

print(f"Dimensiones finales: {df_gold.shape}")
print(f"Columnas: {df_gold.columns}")
print()
df_gold.head(10)

---
## 3. Validacion contra Metricas Originales

Se verifica que el dataset Gold reproduce exactamente los resultados del pipeline
monolitico original. Esto garantiza que la refactorizacion en capas no altero
la logica de procesamiento.

In [ ]:
# =============================================================================
# 3.1 Validacion dimensional
# =============================================================================

actual_rows = df_gold.shape[0]
actual_cols = df_gold.shape[1]

print("=" * 70)
print("VALIDACION CONTRA METRICAS ORIGINALES")
print("=" * 70)

checks = {
    'Filas': (actual_rows, EXPECTED_ROWS, actual_rows == EXPECTED_ROWS),
    'Columnas': (actual_cols, EXPECTED_COLS, actual_cols == EXPECTED_COLS),
    'Nombres de columnas': (
        str(df_gold.columns), str(EXPECTED_COLUMNS),
        df_gold.columns == EXPECTED_COLUMNS
    ),
}

all_pass = True
for check_name, (actual, expected, passed) in checks.items():
    status = 'OK' if passed else 'FALLO'
    print(f"  [{status}] {check_name}: esperado={expected}, obtenido={actual}")
    if not passed:
        all_pass = False

print(f"\n  Resultado global: {'TODAS LAS VALIDACIONES PASARON' if all_pass else 'HAY FALLOS'}")

In [ ]:
# =============================================================================
# 3.2 Validacion de la estacion top
# =============================================================================

top_station = (
    df_gold
    .group_by('Boarding_stop_stn')
    .agg(pl.col('passenger_count').sum().alias('total'))
    .sort('total', descending=True)
    .head(1)
)

actual_top = top_station['Boarding_stop_stn'][0]
actual_top_count = top_station['total'][0]
top_match = actual_top == EXPECTED_TOP_STN

print(f"  [{'OK' if top_match else 'FALLO'}] Top estacion: "
      f"esperada={EXPECTED_TOP_STN}, obtenida={actual_top} "
      f"({actual_top_count:,} pasajeros)")

In [ ]:
# =============================================================================
# 3.3 Tabla de metricas Gold
# =============================================================================

n_stations = df_gold['Boarding_stop_stn'].n_unique()
n_bins     = df_gold['time_bin_15min'].n_unique()
date_min   = df_gold['time_bin_15min'].min()
date_max   = df_gold['time_bin_15min'].max()

gold_metrics = pd.DataFrame({
    'Metrica': [
        'Filas (combinaciones estacion x ventana)',
        'Columnas',
        'Estaciones unicas',
        'Ventanas temporales unicas',
        'Cobertura temporal inicio',
        'Cobertura temporal fin',
        'Top estacion por demanda',
        'Passenger count (media)',
        'Passenger count (mediana)',
        'Passenger count (max)',
    ],
    'Valor': [
        f'{actual_rows:,}',
        str(actual_cols),
        f'{n_stations:,}',
        f'{n_bins:,}',
        str(date_min),
        str(date_max),
        f'{actual_top} ({actual_top_count:,} pasajeros)',
        f'{df_gold["passenger_count"].mean():.2f}',
        f'{df_gold["passenger_count"].median():.0f}',
        f'{df_gold["passenger_count"].max():,}',
    ]
})

print("=" * 70)
print("METRICAS DE LA CAPA GOLD")
print("=" * 70)
print(gold_metrics.to_string(index=False))

---
## 4. Evidencia Visual

Se generan graficos para validar visualmente que los patrones espacio-temporales
se preservan tras la agregacion, y para documentar las caracteristicas del dataset
Gold que alimentara los modelos de ML.

In [ ]:
# =============================================================================
# 4.1 Top 10 estaciones por demanda total
# =============================================================================

top10 = (
    df_gold
    .group_by('Boarding_stop_stn')
    .agg(pl.col('passenger_count').sum().alias('total'))
    .sort('total', descending=True)
    .head(10)
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(
    top10['Boarding_stop_stn'], top10['total'],
    color=sns.color_palette('viridis', 10),
    edgecolor='black', linewidth=0.5
)
for bar in bars:
    w = bar.get_width()
    ax.text(w + top10['total'].max() * 0.015, bar.get_y() + bar.get_height() / 2,
            f'{w:,.0f}', va='center', fontsize=10)

ax.set_xlabel('Total de Pasajeros')
ax.set_ylabel('Estacion')
ax.set_title('Capa Gold — Top 10 Estaciones por Demanda Total (Ventanas 15 min)',
             fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 4.2 Demanda promedio por hora (validacion del patron temporal)
# =============================================================================

hourly_agg = (
    df_gold
    .group_by('hour')
    .agg(pl.col('passenger_count').mean().alias('avg_passengers'))
    .sort('hour')
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(
    hourly_agg['hour'], hourly_agg['avg_passengers'],
    color='steelblue', edgecolor='black', linewidth=0.5, alpha=0.8
)
ax.set_xlabel('Hora del Dia')
ax.set_ylabel('Promedio de Pasajeros por Ventana de 15 min')
ax.set_title('Capa Gold — Demanda Promedio por Hora (Validacion del Patron Temporal)',
             fontsize=14, fontweight='bold')
ax.set_xticks(range(5, 23))
ax.set_xticklabels([f'{h:02d}:00' for h in range(5, 23)], rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 4.3 Distribucion del conteo de pasajeros por ventana
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 5))
passenger_counts = df_gold['passenger_count'].to_pandas()

ax.hist(passenger_counts, bins=50, color='steelblue', edgecolor='black',
        linewidth=0.5, alpha=0.8)
ax.axvline(passenger_counts.mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Media = {passenger_counts.mean():.2f}')
ax.axvline(passenger_counts.median(), color='orange', linestyle='--', linewidth=1.5,
           label=f'Mediana = {passenger_counts.median():.0f}')

ax.set_xlabel('Pasajeros por Ventana de 15 min')
ax.set_ylabel('Frecuencia')
ax.set_title('Capa Gold — Distribucion del Conteo de Pasajeros por Ventana',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

---
## 5. Exportacion

Se exporta el dataset Gold en dos formatos:

- **Parquet** (Snappy): formato optimizado para pipelines de ML y lectura columnar.
- **CSV**: formato compatible con la implementacion concurrente en Go (PC2),
  que puede no tener soporte nativo de Parquet sin dependencias adicionales.

In [ ]:
# =============================================================================
# 5.1 Exportacion a Parquet (capa Gold)
# =============================================================================

df_gold.write_parquet(str(GOLD_PARQUET), compression='snappy', use_pyarrow=True)
parquet_mb = GOLD_PARQUET.stat().st_size / (1024 * 1024)

print("=" * 70)
print("CAPA GOLD — EXPORTACION COMPLETADA")
print("=" * 70)
print(f"  Ruta Parquet: {GOLD_PARQUET}")
print(f"  Tamano:       {parquet_mb:.2f} MB")

In [ ]:
# =============================================================================
# 5.2 Exportacion a CSV (compatibilidad con Go)
# =============================================================================

df_gold.write_csv(str(GOLD_CSV))
csv_mb = GOLD_CSV.stat().st_size / (1024 * 1024)

print(f"  Ruta CSV:     {GOLD_CSV}")
print(f"  Tamano CSV:   {csv_mb:.2f} MB")
print(f"  Ratio Parquet/CSV: {parquet_mb / csv_mb:.2f}x")

In [ ]:
# =============================================================================
# 5.3 Copia a data/processed/ (compatibilidad con pipeline anterior)
# =============================================================================
# Se mantiene una copia en data/processed/ para que las referencias existentes
# en el README y en el pipeline de Go (PC2) sigan funcionando.

import shutil

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
shutil.copy2(str(GOLD_PARQUET), str(DATA_PROCESSED / 'dataset_clean_15min.parquet'))
shutil.copy2(str(GOLD_CSV),     str(DATA_PROCESSED / 'dataset_clean_15min.csv'))

print(f"  Copia en data/processed/: OK")

In [ ]:
# =============================================================================
# 6. Verificacion de integridad del archivo exportado
# =============================================================================

df_verify = pl.read_parquet(str(GOLD_PARQUET))

assert df_verify.shape == df_gold.shape, \
    f"Dimensiones: {df_verify.shape} != {df_gold.shape}"
assert df_verify.columns == df_gold.columns, \
    f"Columnas: {df_verify.columns} != {df_gold.columns}"

# Validacion cruzada con data/processed (debe ser identico)
df_processed = pl.read_parquet(str(DATA_PROCESSED / 'dataset_clean_15min.parquet'))
assert df_processed.shape == df_gold.shape, \
    "data/processed/ difiere de data/gold/"

print("Verificacion de integridad: EXITOSA")
print(f"  Gold:      {df_verify.shape}")
print(f"  Processed: {df_processed.shape} (identico)")
print(f"\nEsquema del archivo Parquet Gold:")
for col_name, col_dtype in zip(df_verify.columns, df_verify.dtypes):
    print(f"  {col_name:<25s} -> {str(col_dtype)}")

print(f"\n>>> Pipeline medallon completado: Bronze -> Silver -> Gold")

---
## Resumen del Pipeline Medallon Completo

| Capa | Filas | Columnas | Archivo | Tamano |
|------|------:|--------:|---------|--------|
| Bronze | 2,000,000 | 13 | `data/bronze/bus_data_oct2017_raw.parquet` | ~30 MB |
| Silver | 1,952,152 | 16 | `data/silver/bus_data_oct2017_clean.parquet` | ~61 MB |
| Gold | 506,545 | 8 | `data/gold/dataset_clean_15min.parquet` | ~1.5 MB |

### Trazabilidad de la Reduccion de Datos

```
CSV crudo (~11 GB) --scan_csv--> Bronze (2,000,000) --limpieza--> Silver (1,952,152) --agregacion--> Gold (506,545)
                                                       -47,848                          x0.26 (compresion por group_by)
                                                       (2.39%)                          
```

### Referencias

- Zou, X. et al. (2022). *Passenger Flow Prediction Using Smart Card Data.* Wireless Communications and Mobile Computing.
- Hao, S. et al. (2019). *MGC-RNN for Short-Term Forecasting of Transit Passenger Flow.*
- Liu, Y. et al. (2020). *Short-term OD demand prediction in urban rail transit.*
- DST-TransitNet (2023). *Dynamic Spatio-Temporal Model for Station-Level Transit Ridership Prediction.*